In [2]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao


,Ano,Base,Tabelas
0,2025,2025.duckdb,9
1,2026,2026.duckdb,10


In [3]:
# ==========================
# ENTREGAS MACHADO E BRITES
# Ano completo - sem filtro de mês
# ==========================

ANO = 2026

df_entregas = CON[ANO].execute(
    f"""
    SELECT DISTINCT
        FENTREGA,
        CODEUT

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND TRANSPORTISTA ILIKE '%MACHADO%'
      AND TRANSPORTISTA ILIKE '%BRITES%'

    ORDER BY
        FENTREGA,
        CODEUT
    """,
    [GESTAO]
).df()

# Visualizar resultados
print(f"ANO: {ANO}")
print(f"TOTAL DE ENTREGAS: {len(df_entregas)}")

with pd.option_context("display.max_rows", None):
    display(df_entregas)

ANO: 2026
TOTAL DE ENTREGAS: 15


,FENTREGA,CODEUT
0,20260806,3514019
1,20260807,3514775
2,20260810,3515749
3,20260811,3517262
4,20260811,3518568
5,20260812,3518143
6,20260812,3519571
7,20260813,3519221
8,20260814,3520565
9,20260817,3521477


In [8]:
# ==================================================
# ALPRO (311) - SALVESEN 2 -> AUCHAN LOGÍSTICA
# AGOSTO, SETEMBRO E OUTUBRO DE 2025
# ==================================================

import pandas as pd

ANO = 2025
CLIENTE = "311"
MESES = ["202508", "202509", "202510"]

# 1. Identificar colunas da base de dados
colunas = CON[ANO].execute(
    f"DESCRIBE inform_27_{ANO}"
).df()["column_name"].tolist()

# 2. Identificar campos de origem e destino
campos_origem = [
    c for c in colunas
    if any(x in c.upper() for x in ["ORIG", "EXPED", "REMET"])
]

campos_destino = [
    c for c in colunas
    if any(x in c.upper() for x in ["DEST", "CHEG", "LLEG", "RECEP"])
]

print("Campos de origem:", campos_origem)
print("Campos de destino:", campos_destino)

if not campos_origem or not campos_destino:
    raise ValueError(
        "Não foi possível identificar os campos de origem e destino. "
        "Verifica os nomes apresentados na estrutura da tabela."
    )

# 3. Construir condições de pesquisa
filtro_origem = " OR ".join(
    f'CAST("{c}" AS VARCHAR) ILIKE ?'
    for c in campos_origem
)

filtro_destino = " OR ".join(
    f'CAST("{c}" AS VARCHAR) ILIKE ?'
    for c in campos_destino
)

# 4. Preparar parâmetros
parametros = (
    [GESTAO]
    + MESES
    + [CLIENTE]
    + ["%SALVESEN%"] * len(campos_origem)
    + ["%AUCHAN%"] * len(campos_destino)
)

# 5. Executar consulta
df_auchan = CON[ANO].execute(
    f"""
    SELECT *

    FROM inform_27_{ANO}

    WHERE Gestion = ?

      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
          IN (?, ?, ?)

      AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') = ?

      AND ({filtro_origem})

      AND ({filtro_destino})

    ORDER BY FENTREGA, REFERENCIA
    """,
    parametros
).df()

# 6. Converter valores numéricos
for coluna in ["PALETS", "PESO_BRUTO", "INGRESODT", "COSTEDT"]:
    if coluna in df_auchan.columns:
        df_auchan[coluna] = pd.to_numeric(
            df_auchan[coluna].astype("string")
            .str.replace(",", ".", regex=False),
            errors="coerce"
        )

# 7. Selecionar colunas relevantes
colunas_resultado = [
    "FENTREGA",
    "CODACT",
    "ACTIVIDAD",
    "REFERENCIA",
    "CODEUT",
    *campos_origem,
    *campos_destino,
    "PALETS",
    "PESO_BRUTO"
]

colunas_resultado = list(dict.fromkeys(
    c for c in colunas_resultado
    if c in df_auchan.columns
))

df_auchan = df_auchan[colunas_resultado]

# 8. Resultados
print(f"\nTotal de registos encontrados: {len(df_auchan):,}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(df_auchan)

Campos de origem: ['LOCORIGEN', 'PROV_ORIGEN', 'PAISORIGEN', 'ORIGEN', 'ficheiro_origem']
Campos de destino: ['LOCDESTINO', 'PROV_DESTINO', 'PAISDESTINO', 'DESTINO']

Total de registos encontrados: 50


,FENTREGA,CODACT,ACTIVIDAD,REFERENCIA,CODEUT,LOCORIGEN,PROV_ORIGEN,PAISORIGEN,ORIGEN,ficheiro_origem,LOCDESTINO,PROV_DESTINO,PAISDESTINO,DESTINO,PALETS,PESO_BRUTO
0,20250801,311,ALPRO PORTUGAL,5020526774 412772919,3211327,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),2,1434.19
1,20250804,311,ALPRO PORTUGAL,5020564859 412814751,3212324,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Valongo,Porto,PORTUGAL,AUCHAN RETAIL PORT.II-VALONGO / Alpro Portugal...,1,122.98
2,20250805,311,ALPRO PORTUGAL,5020552397 412811536,3214160,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),24,18833.49
3,20250805,311,ALPRO PORTUGAL,5020564802 412801888,3214064,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Alverca,Lisboa,PORTUGAL,AUCHAN RETAIL PORT.II-ALVERCA / Alpro Portugal...,1,321.2
4,20250805,311,ALPRO PORTUGAL,5020564858 412814750,3214160,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),3,1941.9
5,20250807,311,ALPRO PORTUGAL,5020579809 412837969,3216479,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),19,14727.93
6,20250808,311,ALPRO PORTUGAL,5020596925 412849561,3217758,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),2,903.58
7,20250808,311,ALPRO PORTUGAL,5020596926 412849561,3217758,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Azambuja,Lisboa,PORTUGAL,AUCHAN AZAMBUJA / Alpro Portugal (311),1,25.89
8,20250808,311,ALPRO PORTUGAL,5020599612 412842331,3217759,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Alverca,Lisboa,PORTUGAL,AUCHAN RETAIL PORT.II-ALVERCA / Alpro Portugal...,1,616.07
9,20250811,311,ALPRO PORTUGAL,5020594381 412845633,3218876,Vila Nova da Rainha,Lisboa,PORTUGAL,SALVESEN LOGISTICA AZABUJA 2 / Alpro Portugal ...,SAL_DAT027_2025_07_12.csv,Alverca,Lisboa,PORTUGAL,AUCHAN RETAIL PORT.II-ALVERCA / Alpro Portugal...,3,306.14


In [9]:
# ==================================================
# ALPRO (311) - AUCHAN LOGÍSTICA
# DIAS COM ENTREGAS POR SEMANA
# Agosto, Setembro e Outubro de 2025
# ==================================================

import pandas as pd

# 1. Preparar datas
df_dias = df_auchan.copy()

df_dias["DATA"] = pd.to_datetime(
    df_dias["FENTREGA"].astype("string").str.replace(
        r"\.0$", "", regex=True
    ),
    format="%Y%m%d",
    errors="coerce"
)

# 2. Considerar apenas segunda-feira a sábado
df_dias = df_dias[
    df_dias["DATA"].notna()
    & (df_dias["DATA"].dt.dayofweek <= 5)
].copy()

# 3. Identificar o início de cada semana (segunda-feira)
df_dias["INICIO_SEMANA"] = (
    df_dias["DATA"]
    - pd.to_timedelta(df_dias["DATA"].dt.dayofweek, unit="D")
)

# 4. Contar dias distintos com entregas por semana
df_semanas = (
    df_dias.groupby("INICIO_SEMANA")
    .agg(
        DIAS_COM_ENTREGAS=("DATA", "nunique"),
        REGISTOS=("DATA", "size")
    )
    .reset_index()
)

# 5. Identificar o sábado de cada semana
df_semanas["FIM_SEMANA"] = (
    df_semanas["INICIO_SEMANA"] + pd.Timedelta(days=5)
)

# 6. Calcular dias sem entregas
df_semanas["DIAS_POSSIVEIS"] = 6

df_semanas["DIAS_SEM_ENTREGAS"] = (
    df_semanas["DIAS_POSSIVEIS"]
    - df_semanas["DIAS_COM_ENTREGAS"]
)

# 7. Organizar colunas
df_semanas = df_semanas[
    [
        "INICIO_SEMANA",
        "FIM_SEMANA",
        "DIAS_COM_ENTREGAS",
        "DIAS_SEM_ENTREGAS",
        "REGISTOS"
    ]
]

# 8. Resultados
display(df_semanas)

print(f"Semanas com entregas: {len(df_semanas)}")
print(f"Dias distintos com entregas: {df_dias['DATA'].nunique()}")

if not df_semanas.empty:
    print(
        f"Média de dias com entregas por semana: "
        f"{df_semanas['DIAS_COM_ENTREGAS'].mean():.2f}"
    )

,INICIO_SEMANA,FIM_SEMANA,DIAS_COM_ENTREGAS,DIAS_SEM_ENTREGAS,REGISTOS
0,2025-07-28,2025-08-02,1,5,1
1,2025-08-04,2025-08-09,4,2,8
2,2025-08-11,2025-08-16,4,2,10
3,2025-08-18,2025-08-23,1,5,1
4,2025-09-22,2025-09-27,1,5,3
5,2025-09-29,2025-10-04,3,3,6
6,2025-10-06,2025-10-11,3,3,9
7,2025-10-13,2025-10-18,3,3,4
8,2025-10-20,2025-10-25,2,4,5
9,2025-10-27,2025-11-01,2,4,3


Semanas com entregas: 10
Dias distintos com entregas: 24
Média de dias com entregas por semana: 2.40
